# Metadata support

In [ ]:

import os
import numpy as np
import cupy as cp
from matplotlib import pyplot as plt


Setting resource folder

In [ ]:
resources_dir = os.getenv("PYNVIMGCODEC_EXAMPLES_RESOURCES_DIR", "../assets/images/")

Import nvImageCodec module and create Decoder

In [ ]:
#os.environ['PYNVIMGCODEC_VERBOSITY'] = '5' #uncomment for verbose log output

In [ ]:
from nvidia import nvimgcodec
decoder = nvimgcodec.Decoder()
nv_img = decoder.read(resources_dir + "cat-1046544_640.jp2")

Define function to extract and print metadata for all sub-images

In [ ]:
def print_metadata(cs):
    print("Number of images:", cs.num_images)
    for code_stream_idx in range(0, cs.num_images):
        scs = cs.get_sub_code_stream(code_stream_idx)
        metadata = decoder.get_metadata(scs)
        if len(metadata) == 0:
            print(f"No metadata for image {code_stream_idx}")
            print("="*50)
        else:
            print(f"Metadata for image {code_stream_idx}:")
            print("="*50)
            for m in metadata:
                print(m)
                if m.kind != nvimgcodec.MetadataKind.ICC_PROFILE:
                    print(" "*5, m.buffer.decode('utf-8'))
                else:
                    print(f"{' ' * 5}{m.buffer[:200]}")


Define function to decode and show all sub-images

In [ ]:
def decode_and_show(cs):
    for code_stream_idx in range(0, cs.num_images):
        scs = cs.get_sub_code_stream(code_stream_idx)
        img = decoder.decode(scs)
        plt.figure()
        plt.title(f"Image {code_stream_idx}")
        plt.imshow(img.cpu())
        plt.show()


## Geo

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Alex_2016-01-14_1300Z_(Geotiff).tif")))
print_metadata(cs)

Decode image

In [ ]:
decode_and_show(cs)

## Aperio

Aorta tissue, brightfield, JPEG 2000, YCbCr

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "JP2K-33003-1.svs")))
print_metadata(cs)

Decode all subimages

In [ ]:
decode_and_show(cs)

## Philips

Lymph node section, H&E stain, brightfield, BigTIFF, barcode attribute, from CAMELYON16 data set

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Philips-1.tiff")))
print_metadata(cs)

In [ ]:
decode_and_show(cs)

## Leica

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Leica-Fluorescence-1.scn")))
print_metadata(cs)

In [ ]:
decode_and_show(cs)

## Trestle

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "CMU-1.tif")))
print_metadata(cs)

In [ ]:
decode_and_show(cs)

## Ventana

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Ventana-1.bif")))
print_metadata(cs)

In [ ]:
decode_and_show(cs)

## Generic tiff tag reading

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Ventana-1.bif")))
image_description_tag_id = 270
for code_stream_idx in range(0, cs.num_images):
    scs = cs.get_sub_code_stream(code_stream_idx)

    img_description = decoder.get_metadata(scs, id = image_description_tag_id).value
    print(f"Image #{code_stream_idx} description: {img_description}")

## List all available TIFF tags

Use `MetadataKind.TIFF_TAG_LIST` to discover which tags are available in an image, then fetch each one individually.

In [ ]:
cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Ventana-1.bif")))

# List all available TIFF tags
tag_list_metadata = decoder.get_metadata(cs, kind=nvimgcodec.MetadataKind.TIFF_TAG_LIST)
tag_ids = tag_list_metadata.value
print(f"Available tags: {tag_ids}")

# Iterate and fetch each tag
for tag_id in tag_ids:
    tag = decoder.get_metadata(cs, id=tag_id)
    if tag is not None:
        print(f"  Tag {tag_id}: {tag.value}")

# Non-existent tags return None
tag = decoder.get_metadata(cs, id=65535)
assert tag is None
print("Non-existent tag lookup returned None")

## Use `PIL.TiffTags` together with nvImageCodec generic Tiff tag reading

In [ ]:
! pip install pillow

In [ ]:
from PIL.TiffTags import TAGS

cs = nvimgcodec.CodeStream(os.path.abspath(os.path.join(resources_dir, "Ventana-1.bif")))
for code_stream_idx in range(0, cs.num_images):
    scs = cs.get_sub_code_stream(code_stream_idx)
    for tag_id, tag_name in TAGS.items():
        if not isinstance(tag_id, int):
            continue
        tag_metadata = decoder.get_metadata(scs, id=tag_id)
        if tag_metadata is not None:
            print(f"Image #{code_stream_idx} {tag_name}: {tag_metadata.value}")